#### Notebook for Converting between Noise and Velocity Prediction

We'll assume the latent diffusion, aka diffusion in latent space rather than pixel space, (https://arxiv.org/abs/2112.10752) here.

The forward diffusion process creates noised latents

$$x_t = \alpha_t \cdot x_0 + \sigma_t \cdot \epsilon$$. The velocity of a point mass it its time derivative.
$$v_t = \dot{\alpha_t} x_0 + \dot{\sigma_t} \epsilon$$

Diffusion inference--$\hat{x}_0$, the empiricla pure data mass, is solved for via $$x_t = \alpha_t \cdot x_0 + \sigma_t \cdot \epsilon$$, and substituted into the velocity.

$$v_t = \dot{\alpha_t} \cdot \frac{x_t - \sigma_t \epsilon}{\alpha_t} + \dot{\sigma_t} \epsilon$$

$$v_t = \frac{\dot{\alpha_t}}{\alpha_t} x_t + (\dot{\sigma_t} - \frac{\alpha_t \sigma_t}{\alpha_t}) \epsilon$$

For this notebook we'll be interested converting the noise predictor of an sd1.x (such as sd1.5) into a velocity predictor.

DDIM (Song et al. 2020) the reverse process can be made deterministic and that you can skip timesteps without retraining, because the marginals $$q(x_t∣x_0)$$. In practice SD 1.5 is almost always run with 20–50 DDIM steps.

#### Let's first load a pretrained sd1.5 model

Let's first load a pretrained sd1.5 model and sampling from it.

The "diffusion" happens in latent space, so--

x_0 (image) → VAE encode → z_0 (latent) → add noise → z_t → UNet → predicted noise in latent space

The conversion we'll do to a velocity prediction. 

Since the decoder takes as input the reconstructed latent of x_0, for velocity prediction, simply solve for $\hat{x_0}$ from the velocity prediction and .decode that. 

You can safely ignore the text_model.embeddings.position_ids UNEXPECTED warning.

In [1]:
from diffusers import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
import torch

model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae").to("mps")
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to("mps")
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet").to("mps")

torch.set_grad_enabled(False)

/Users/annhe/anaconda3/envs/glass_flows/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: stable-diffusion-v1-5/stable-diffusion-v1-5
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [2]:
"""
def encode(image_tensor):
    # image_tensor: (1, 3, 512, 512), values in [-1, 1]
    with torch.no_grad():
        latent = vae.encode(image_tensor).latent_dist.sample()
        latent = latent * vae.config.scaling_factor  # scale to unit variance
    return latent  # (1, 4, 64, 64)

def decode(latent):
    with torch.no_grad():
        latent = latent / vae.config.scaling_factor
        image = vae.decode(latent).sample  # (1, 3, 512, 512)
    return image
"""

'\ndef encode(image_tensor):\n    # image_tensor: (1, 3, 512, 512), values in [-1, 1]\n    with torch.no_grad():\n        latent = vae.encode(image_tensor).latent_dist.sample()\n        latent = latent * vae.config.scaling_factor  # scale to unit variance\n    return latent  # (1, 4, 64, 64)\n\ndef decode(latent):\n    with torch.no_grad():\n        latent = latent / vae.config.scaling_factor\n        image = vae.decode(latent).sample  # (1, 3, 512, 512)\n    return image\n'

In [3]:
_TEXT_PROMPT = "A visual of a woman on vacation in Bali."

vae          # image ↔ latent \
text_encoder # prompt → (1, 77, 768) \
tokenizer    # prompt → token ids \
unet         # (z_t, t, text_embeddings) → noise in latent space

In [4]:
# for sd1.5, time is an integer between 0 and 999, indexing into the 1000 step ddpm schedule
t = 980
t_tensor = torch.tensor([t], device="mps") 

In [5]:
from diffusers import DDIMScheduler
import torch

scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")

def encode_text(prompt):
    tokens = tokenizer(
        prompt,
        return_tensors="pt",
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
    )
    return text_encoder(tokens.input_ids.to("mps")).last_hidden_state

def sample_ddpm_latent(time: int, batch_size: int = 1):
    """
    Sample a noisy latent z_t ~ q(z_t) at DDPM timestep t (0-999).
    Returns a latent you can pass directly to unet(z_t, t, text_embeddings).
    """
    alpha_t = scheduler.alphas_cumprod[time].sqrt()
    sigma_t = (1 - scheduler.alphas_cumprod[time]).sqrt()

    # pure noise in latent space
    epsilon = torch.randn(batch_size, 4, 64, 64, device="mps")

    # z_t = alpha_t * 0 + sigma_t * epsilon
    # (x_0 term drops out since we have no image to noise — we're sampling from the prior)
    z_t = sigma_t * epsilon

    return z_t

In [6]:
z_t = sample_ddpm_latent(time=t)
noise_pred = unet(z_t, t_tensor, encoder_hidden_states=encode_text(_TEXT_PROMPT))

In [ ]:
"""
def noise_to_velocity(noise_prediction, alpha_t, sigma_t):
"""

In [ ]:
"""
sampling via epsilon prediction in latent space
"""

In [ ]:
"""
sampling via velocity prediction in latent space
"""